# EXP-007: Ridge Meta-Learner Stack + 0.30/0.70 Blend with PF Heuristic

**⚡ RUNS ON: Kaggle GPU (2×T4) — accelerator must be ON**

| Item | Value |
|------|-------|
| Target env | Kaggle 2×T4 GPU |
| Compute | ~60 min full run |
| Expected gain | Largest single lever — LB 9.964 → 7.7–7.9 target |
| Risk | Medium — koolbox unavailable; uses our CV wrapper |

## Architecture (from rogii-ridge-sp45-proj.ipynb, LB 7.748)
1. Train 3 LGB + 3 CB base models with 5-fold GroupKFold → OOF matrix (N × 6)
2. Ridge meta-learner on OOF matrix → `ridge_stack_pred`
3. Model-free PF/beam heuristic → `heuristic_pred`
4. `final = 0.30 * ridge_stack_pred + 0.70 * heuristic_pred`

## Ablation order
A. Heuristic only | B. Ridge stack only | C. Blend 0.30/0.70 | D. Tune blend ratio

In [ ]:
import sys
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

# ── Config ────────────────────────────────────────────────────────
EXP_ID      = 'EXP-007'
TINY_SAMPLE = False   # True = first 10 wells only (sanity check, still on Kaggle)
N_FOLDS     = 5
SEED        = 42
BLEND_ALPHA = 0.30    # Ridge weight; 0.70 on heuristic

# ── Paths (Kaggle only) ───────────────────────────────────────────
INPUT_DIR = Path('/kaggle/input/rogii')
WORK_DIR  = Path('/kaggle/working')
SRC_DIR   = Path('/kaggle/input/rogii-src')  # src/ uploaded as Kaggle dataset
WORK_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR))

print(f'EXP_ID      = {EXP_ID}')
print(f'TINY_SAMPLE = {TINY_SAMPLE}')
print(f'BLEND_ALPHA = {BLEND_ALPHA} Ridge / {1-BLEND_ALPHA} heuristic')
print(f'GPU available: {cb.utils.get_gpu_device_count()} GPU(s)')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────
train = pd.read_csv(INPUT_DIR / 'train.csv')
test  = pd.read_csv(INPUT_DIR / 'test.csv')

if TINY_SAMPLE:
    sample_wells = train['well_code'].unique()[:10]
    train = train[train['well_code'].isin(sample_wells)].copy()
    test  = test[test['well_code'].isin(sample_wells)].copy()
    print(f'TINY_SAMPLE: {len(sample_wells)} wells')

print(f'Train: {train.shape}  Test: {test.shape}')
print(f'Train wells: {train["well_code"].nunique()}  Test wells: {test["well_code"].nunique()}')

# EXP-011 prep: check same-well overlap
overlap = set(train['well_code'].unique()) & set(test['well_code'].unique())
print(f'\nTrain/test well overlap: {len(overlap)} wells  ← save to open_questions.md')
if overlap:
    print(f'  Overlapping wells: {sorted(overlap)[:10]}{"..." if len(overlap) > 10 else ""}')

In [ ]:
# ── Feature preparation ───────────────────────────────────────────
# Load pre-built 150+ feature matrix from rogii-eda-features.ipynb artifact.
# If not found, falls back to raw numeric columns (placeholder only).

FEATURE_STORE     = WORK_DIR / 'features_train.npy'
FEATURE_COLS_PATH = WORK_DIR / 'feature_cols.json'

if FEATURE_STORE.exists():
    X_train = np.load(FEATURE_STORE)
    X_test  = np.load(WORK_DIR / 'features_test.npy')
    with open(FEATURE_COLS_PATH) as f:
        FEATURE_COLS = json.load(f)
    print(f'Loaded pre-built features: {X_train.shape}')
else:
    print('WARNING: No pre-built feature store — run rogii-eda-features.ipynb first.')
    print('Falling back to raw numeric columns (placeholder).')
    FEATURE_COLS = [c for c in train.columns
                    if c not in ['well_code', 'TVT', 'is_eval', 'row_id']
                    and train[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
    X_train = train[FEATURE_COLS].fillna(0).values
    X_test  = test[FEATURE_COLS].fillna(0).values
    print(f'Placeholder features: {X_train.shape}')

eval_mask   = train['is_eval'].astype(bool)
X_eval      = X_train[eval_mask]
y_eval      = train.loc[eval_mask, 'TVT'].values
groups_eval = train.loc[eval_mask, 'well_code'].values
print(f'Eval rows: {X_eval.shape}  Target: {y_eval.shape}')

In [ ]:
# ── Base model configs (GPU) ──────────────────────────────────────
LGB_BASE = {'objective': 'regression', 'metric': 'rmse', 'verbosity': -1,
            'n_jobs': -1, 'seed': SEED, 'device': 'gpu'}
LGB_CONFIGS = [
    # LGB-1: aw pipeline baseline
    {**LGB_BASE, 'num_leaves': 255, 'learning_rate': 0.02,    'reg_lambda': 3.0,
     'n_estimators': 5000,  'subsample': 0.8,   'colsample_bytree': 0.8},
    # LGB-2: medium complexity
    {**LGB_BASE, 'num_leaves': 127, 'learning_rate': 0.015,   'reg_lambda': 10.0,
     'n_estimators': 7000,  'subsample': 0.7,   'colsample_bytree': 0.7},
    # LGB-3: Optuna heavy-regularization (rogii-sel15, H-011)
    {**LGB_BASE, 'num_leaves': 64,  'learning_rate': 0.00935, 'reg_alpha': 10.79,
     'reg_lambda': 95.75, 'colsample_bytree': 0.393, 'min_child_samples': 40,
     'subsample': 0.474, 'n_estimators': 10000},
]

CB_BASE = {'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'verbose': 0,
           'random_seed': SEED, 'task_type': 'GPU'}
CB_CONFIGS = [
    # CB-1: catboost-2 from aw blend (weight 0.272)
    {**CB_BASE, 'iterations': 5000, 'learning_rate': 0.03, 'depth': 8,
     'l2_leaf_reg': 3.0, 'subsample': 0.8},
    # CB-2: catboost-3 from aw blend (weight 0.376)
    {**CB_BASE, 'iterations': 5000, 'learning_rate': 0.02, 'depth': 10,
     'l2_leaf_reg': 5.0, 'subsample': 0.75},
    # CB-3: lower lr, deeper
    {**CB_BASE, 'iterations': 8000, 'learning_rate': 0.01, 'depth': 10,
     'l2_leaf_reg': 8.0, 'subsample': 0.7},
]
print(f'{len(LGB_CONFIGS)} LGB + {len(CB_CONFIGS)} CB = {len(LGB_CONFIGS)+len(CB_CONFIGS)} base models')

In [ ]:
# ── Train base models → collect OOF matrix ────────────────────────
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

kf      = GroupKFold(n_splits=N_FOLDS)
splits  = list(kf.split(X_eval, groups=groups_eval))
n_models    = len(LGB_CONFIGS) + len(CB_CONFIGS)
oof_matrix  = np.zeros((len(X_eval), n_models))
test_matrix = np.zeros((len(X_test),  n_models))
model_scores = []
t_total = time.time()

for m_idx, (model_type, configs) in enumerate([('LGB', LGB_CONFIGS), ('CB', CB_CONFIGS)]):
    for cfg_idx, cfg in enumerate(configs):
        col   = (3 * (model_type == 'CB')) + cfg_idx
        label = f'{model_type}-{cfg_idx+1}'
        print(f'\n── {label} ──')
        t0 = time.time()
        oof_fold  = np.zeros(len(X_eval))
        test_fold = np.zeros(len(X_test))

        for fold, (tr_idx, val_idx) in enumerate(splits):
            X_tr, X_val = X_eval[tr_idx], X_eval[val_idx]
            y_tr, y_val = y_eval[tr_idx], y_eval[val_idx]
            if model_type == 'LGB':
                model = lgb.LGBMRegressor(**cfg)
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
            else:
                model = cb.CatBoostRegressor(**cfg)
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=200)
            val_pred  = model.predict(X_val)
            test_pred = model.predict(X_test)
            oof_fold[val_idx]  = val_pred
            test_fold         += test_pred / N_FOLDS
            print(f'  fold {fold+1}: RMSE={rmse(y_val, val_pred):.5f}  {time.time()-t0:.0f}s')

        oof_matrix[:, col]  = oof_fold
        test_matrix[:, col] = test_fold
        cv = rmse(y_eval, oof_fold)
        model_scores.append({'model': label, 'cv_rmse': round(cv, 6)})
        print(f'  {label} CV={cv:.5f}')
        np.save(WORK_DIR / f'{EXP_ID}_{label}_oof.npy',  oof_fold)
        np.save(WORK_DIR / f'{EXP_ID}_{label}_test.npy', test_fold)

np.save(WORK_DIR / f'{EXP_ID}_oof_matrix.npy',  oof_matrix)
np.save(WORK_DIR / f'{EXP_ID}_test_matrix.npy', test_matrix)
print(f'\nAll base models: {(time.time()-t_total)/60:.1f} min')
for s in model_scores:
    print(f'  {s["model"]}: {s["cv_rmse"]}')

In [ ]:
# ── Ridge meta-learner (inline — no external import needed) ───────
class RidgeStacker:
    def __init__(self, alpha=1.0, blend_alpha=0.30, seed=42):
        self.alpha = alpha; self.blend_alpha = blend_alpha; self.seed = seed
        self.scaler = StandardScaler(); self.ridge = Ridge(alpha=alpha)
    def fit(self, X, y):
        self.ridge.fit(self.scaler.fit_transform(X), y); return self
    def predict_stack(self, X):
        return self.ridge.predict(self.scaler.transform(X))
    def blend(self, stack, heuristic):
        return self.blend_alpha * stack + (1 - self.blend_alpha) * heuristic
    def fit_predict_oof(self, X, y, cv=5, groups=None):
        oof = np.zeros(len(y))
        kf_i = GroupKFold(cv) if groups is not None else __import__('sklearn.model_selection', fromlist=['KFold']).KFold(cv, shuffle=True, random_state=self.seed)
        splits_ = list(kf_i.split(X, groups=groups) if groups is not None else kf_i.split(X))
        for tr, val in splits_:
            sc = StandardScaler()
            r  = Ridge(alpha=self.alpha)
            r.fit(sc.fit_transform(X[tr]), y[tr])
            oof[val] = r.predict(sc.transform(X[val]))
        self.fit(X, y)
        return oof

stacker   = RidgeStacker(alpha=1.0, blend_alpha=BLEND_ALPHA, seed=SEED)
oof_stack = stacker.fit_predict_oof(oof_matrix, y_eval, cv=N_FOLDS, groups=groups_eval)
test_stack = stacker.predict_stack(test_matrix)
print(f'Ridge stack OOF RMSE: {rmse(y_eval, oof_stack):.5f}')
print(f'Weights: {dict(zip([s["model"] for s in model_scores], stacker.ridge.coef_.round(4)))}')
np.save(WORK_DIR / f'{EXP_ID}_ridge_stack_oof.npy',  oof_stack)
np.save(WORK_DIR / f'{EXP_ID}_ridge_stack_test.npy', test_stack)

In [ ]:
# ── Load PF heuristic ─────────────────────────────────────────────
# Pre-computed 128-seed PF likelihood ensemble from rogii-postprocess-research.ipynb
pf_oof_path = WORK_DIR / 'pf_heuristic_oof.npy'

if pf_oof_path.exists():
    pf_oof  = np.load(pf_oof_path)
    pf_test = np.load(WORK_DIR / 'pf_heuristic_test.npy')
    print(f'PF heuristic OOF RMSE: {rmse(y_eval, pf_oof):.5f}')
else:
    print('WARNING: pf_heuristic_oof.npy not found — using best base model as proxy.')
    best_col = int(np.argmin([rmse(y_eval, oof_matrix[:, c]) for c in range(n_models)]))
    pf_oof   = oof_matrix[:, best_col]
    pf_test  = test_matrix[:, best_col]
    print(f'Proxy (col {best_col}) RMSE: {rmse(y_eval, pf_oof):.5f}')
    print('Re-run after generating PF artifact for true EXP-007 result.')

In [ ]:
# ── Ablation A / B / C / D ────────────────────────────────────────
ablation = {
    'A_heuristic_only':  rmse(y_eval, pf_oof),
    'B_ridge_stack_only': rmse(y_eval, oof_stack),
    'C_blend_030_070':   rmse(y_eval, stacker.blend(oof_stack, pf_oof)),
}
best_alpha, best_d = 0.30, float('inf')
for a in np.arange(0.0, 1.01, 0.05):
    r = rmse(y_eval, a * oof_stack + (1 - a) * pf_oof)
    if r < best_d: best_d, best_alpha = r, a
ablation['D_tuned_blend'] = best_d
ablation['D_best_alpha']  = round(float(best_alpha), 2)

print('=== EXP-007 Ablation ===')
for k, v in ablation.items():
    print(f'  {k}: {v:.5f}' if isinstance(v, float) else f'  {k}: {v}')
print(f'  Baseline (aw pipeline): 10.45212  LB 9.964')

In [ ]:
# ── Final blend + save ────────────────────────────────────────────
final_alpha = best_alpha
final_oof   = final_alpha * oof_stack  + (1 - final_alpha) * pf_oof
final_test  = final_alpha * test_stack + (1 - final_alpha) * pf_test
print(f'Final OOF RMSE (alpha={final_alpha:.2f}): {rmse(y_eval, final_oof):.5f}')
np.save(WORK_DIR / f'{EXP_ID}_final_oof.npy',  final_oof)
np.save(WORK_DIR / f'{EXP_ID}_final_test.npy', final_test)

result = {
    'exp_id': EXP_ID, 'baseline_rmse': 10.45212,
    'ablation': {k: round(float(v), 6) if isinstance(v, float) else v for k, v in ablation.items()},
    'final_rmse': round(rmse(y_eval, final_oof), 6),
    'blend_alpha': final_alpha, 'model_scores': model_scores, 'tiny_sample': TINY_SAMPLE,
}
with open(WORK_DIR / f'{EXP_ID}_result.json', 'w') as f:
    json.dump(result, f, indent=2)
print(json.dumps(result, indent=2))
print('\nNext: record in experiment_queue.md + memory/previous_runs.md')
print('Then: run exp009_projection_postprocess__CPU on top of final_oof/final_test')

In [ ]:
# ── Generate submission ───────────────────────────────────────────
sample_sub = pd.read_csv(INPUT_DIR / 'sample_submission.csv')
sub = sample_sub.copy()
test_sub = test[test['is_eval'].astype(bool)].copy() if 'is_eval' in test.columns else test.copy()
test_sub['TVT'] = final_test
if 'row_id' in test_sub.columns:
    sub = sub.merge(test_sub[['row_id', 'TVT']], on='row_id', how='left', suffixes=('_old', ''))
    if 'TVT_old' in sub.columns:
        sub['TVT'] = sub['TVT'].fillna(sub['TVT_old'])
        sub = sub.drop(columns=['TVT_old'])
sub.to_csv(WORK_DIR / f'{EXP_ID}_submission.csv', index=False)
print(f'Submission: {len(sub)} rows')
sub.head()